# 05_ngram_language_models: N-gram Models on Wikipedia Text
    
This notebook builds an N-gram Language Model from scratch and computes Perplexity metrics using a scraped Wikipedia text corpus.


## 1. Scrape Wikipedia Corpus

In [1]:
import requests
from bs4 import BeautifulSoup
import re

url = "https://en.wikipedia.org/wiki/Natural_language_processing"
resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
soup = BeautifulSoup(resp.content, "html.parser")
paragraphs = [p.get_text().strip() for p in soup.find_all("p") if len(p.get_text().strip()) > 80]
corpus_text = " ".join(paragraphs[:8])

# Normalization
corpus = re.sub(r"[^\w\s]", "", corpus_text).lower().split()
vocab = list(set(corpus))
vocab_size = len(vocab)
print(f"Corpus Tokens: {len(corpus)}, Vocab Size: {vocab_size}")


Corpus Tokens: 350, Vocab Size: 188


### Output Explanation: Raw Corpus Setup
- **Corpus Slicing**: Slices text parameters from the Wikipedia Natural Language Processing page.
- **Vocabulary Size**: The unique vocabulary size ($|V|$) serves as the normalization factor for Laplace smoothing.


## 2. Count Unigrams and Bigrams

In [2]:
from collections import Counter

unigrams = Counter(corpus)
bigrams = Counter(zip(corpus[:-1], corpus[1:]))

print("Top 5 Unigrams:", unigrams.most_common(5))
print("Top 5 Bigrams:", bigrams.most_common(5))


Top 5 Unigrams: [('the', 23), ('of', 14), ('a', 12), ('language', 11), ('and', 10)]
Top 5 Bigrams: [(('natural', 'language'), 9), (('language', 'processing'), 5), (('nlp', 'is'), 4), (('in', 'the'), 4), (('of', 'natural'), 2)]


### Output Explanation: Token Frequencies
- **Frequencies**: Tracks occurrence counts. The most common bigrams represent lexical pairs like `('natural', 'language')` and `('language', 'processing')`.


## 3. Calculate Laplace-Smoothed Transition Probabilities

In [3]:
def get_bigram_prob(w1, w2):
    count_bigram = bigrams[(w1, w2)]
    count_unigram = unigrams[w1]
    # Laplace smoothing formula: (count(w1 w2) + 1) / (count(w1) + |V|)
    return (count_bigram + 1) / (count_unigram + vocab_size)

print("Smoothed Probabilities:")
print("  P(language | natural) =", get_bigram_prob("natural", "language"))
print("  P(methods | natural)  =", get_bigram_prob("natural", "methods"))


Smoothed Probabilities:
  P(language | natural) = 0.050761421319796954
  P(methods | natural)  = 0.005076142131979695


### Output Explanation: Laplace Smoothing Transition
- **Smoothing Effect**: Without smoothing, an unseen bigram like `('natural', 'methods')` would get a probability of $0$. By adding $+1$ to the numerator and $|V|$ to the denominator, we assign it a small, non-zero probability, preventing sequence score saturation.


## 4. Evaluate Sequence Perplexity

In [4]:
import math

test_sequence = ["natural", "language", "processing", "methods", "and", "tasks"]

def compute_perplexity(seq):
    log_prob_sum = 0.0
    for i in range(1, len(seq)):
        w1, w2 = seq[i-1], seq[i]
        prob = get_bigram_prob(w1, w2)
        log_prob_sum += math.log(prob)
        
    avg_log_prob = log_prob_sum / (len(seq) - 1)
    return math.exp(-avg_log_prob)

ppl = compute_perplexity(test_sequence)
print(f"Perplexity of sequence {test_sequence}: {ppl:.4f}")


Perplexity of sequence ['natural', 'language', 'processing', 'methods', 'and', 'tasks']: 86.1401


### Output Explanation: Perplexity Evaluation
- **Perplexity (PPL)**: Measures how well the model predicts the test sequence. A lower perplexity indicates the sequence is more natural and likely according to the bigram probability counts.
